In [1]:
# ============================================================
# Prepare_Import_CSV.ipynb
# Converts synthetic research data to import pipeline format
# Clearly labelled as synthetic — not fake operational data
# ============================================================

import pandas as pd
import numpy as np
import os

print("="*55)
print("  Prepare Import CSV")
print("  SmartTea AI Research Project")
print("="*55)
print()
print("  This notebook converts the synthetic research")
print("  dataset into the controlled import pipeline format.")
print()
print("  The data is clearly labelled as synthetic.")
print("  It goes through the real dual-approval pipeline.")
print()
print("✅ Setup complete!")

  Prepare Import CSV
  SmartTea AI Research Project

  This notebook converts the synthetic research
  dataset into the controlled import pipeline format.

  The data is clearly labelled as synthetic.
  It goes through the real dual-approval pipeline.

✅ Setup complete!


In [2]:
# Load synthetic dataset
df = pd.read_csv('data/tea_demand_timeseries.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['TeaGrade', 'Date'])

print("="*55)
print("  Dataset Loaded")
print("="*55)
print(f"  Rows:   {len(df):,}")
print(f"  Grades: {df['TeaGrade'].unique().tolist()}")
print(f"  From:   {df['Date'].min().date()}")
print(f"  To:     {df['Date'].max().date()}")
print()
for g in sorted(df['TeaGrade'].unique()):
    gdf = df[df['TeaGrade'] == g]
    print(f"  {g:<6}: avg {gdf['DemandKg'].mean():.1f} kg/day")
print()
print("✅ Data loaded!")

  Dataset Loaded
  Rows:   5,475
  Grades: ['BOP', 'BOPF', 'DUST', 'FNGS', 'OP']
  From:   2021-01-01
  To:     2023-12-31

  BOP   : avg 319.0 kg/day
  BOPF  : avg 278.6 kg/day
  DUST  : avg 238.4 kg/day
  FNGS  : avg 178.7 kg/day
  OP    : avg 119.5 kg/day

✅ Data loaded!


In [3]:
# Take last 90 days of data
cutoff = df['Date'].max() - pd.Timedelta(days=90)
df_90  = df[df['Date'] >= cutoff].copy()

print("="*55)
print("  Selected Last 90 Days")
print("="*55)
print(f"  Total rows selected: {len(df_90):,}")
print(f"  From: {df_90['Date'].min().date()}")
print(f"  To:   {df_90['Date'].max().date()}")
print(f"  Days: {df_90['Date'].nunique()}")
print()
for g in sorted(df_90['TeaGrade'].unique()):
    gdf = df_90[df_90['TeaGrade'] == g]
    print(f"  {g:<6}: {len(gdf)} rows  "
          f"avg {gdf['DemandKg'].mean():.1f} kg/day")
print()
print("  Why 90 days?")
print("  - Minimum required for AI is 60 days")
print("  - 90 days gives 30 extra days buffer")
print("  - More history = more reliable forecasting")
print()
print("✅ 90-day selection complete!")

  Selected Last 90 Days
  Total rows selected: 455
  From: 2023-10-02
  To:   2023-12-31
  Days: 91

  BOP   : 91 rows  avg 341.0 kg/day
  BOPF  : 91 rows  avg 290.8 kg/day
  DUST  : 91 rows  avg 247.7 kg/day
  FNGS  : 91 rows  avg 180.3 kg/day
  OP    : 91 rows  avg 112.8 kg/day

  Why 90 days?
  - Minimum required for AI is 60 days
  - 90 days gives 30 extra days buffer
  - More history = more reliable forecasting

✅ 90-day selection complete!


In [4]:
# Convert to import pipeline format
rows = []

for _, r in df_90.iterrows():
    if r['DemandKg'] > 0:
        rows.append({
            'OriginalTransactionDate': r['Date'].strftime(
                                           '%Y-%m-%d'),
            'TeaGrade':                r['TeaGrade'],
            'QuantityKg':              round(r['DemandKg'], 2),
            'UnitOfMeasure':           'kg',
            'TransactionType':         'ProductionOutput',
            'SourceReferenceNumber':   (
                f"SYN-{r['TeaGrade']}-"
                f"{r['Date'].strftime('%Y%m%d')}"),
            'SupplierOrProductionRef': 'SmartTea-Synthetic-Research',
            'WarehouseCode':           'WH-MAIN',
        })

import_df = pd.DataFrame(rows)

print("="*55)
print("  Import Format Created")
print("="*55)
print(f"  Total rows:  {len(import_df):,}")
print(f"  Columns:     {len(import_df.columns)}")
print()
print("  Column names:")
for col in import_df.columns:
    print(f"    {col}")
print()
print("  Sample rows (first 3):")
print()
for i in range(3):
    row = import_df.iloc[i]
    print(f"  Row {i+1}:")
    print(f"    Date:      {row['OriginalTransactionDate']}")
    print(f"    Grade:     {row['TeaGrade']}")
    print(f"    Quantity:  {row['QuantityKg']} kg")
    print(f"    Type:      {row['TransactionType']}")
    print(f"    Reference: {row['SourceReferenceNumber']}")
    print()
print("✅ Import format ready!")

  Import Format Created
  Total rows:  455
  Columns:     8

  Column names:
    OriginalTransactionDate
    TeaGrade
    QuantityKg
    UnitOfMeasure
    TransactionType
    SourceReferenceNumber
    SupplierOrProductionRef
    WarehouseCode

  Sample rows (first 3):

  Row 1:
    Date:      2023-10-02
    Grade:     BOP
    Quantity:  372.4 kg
    Type:      ProductionOutput
    Reference: SYN-BOP-20231002

  Row 2:
    Date:      2023-10-03
    Grade:     BOP
    Quantity:  381.8 kg
    Type:      ProductionOutput
    Reference: SYN-BOP-20231003

  Row 3:
    Date:      2023-10-04
    Grade:     BOP
    Quantity:  384.3 kg
    Type:      ProductionOutput
    Reference: SYN-BOP-20231004

✅ Import format ready!


In [5]:
# Save CSV file
os.makedirs('data', exist_ok=True)

save_path = 'data/synthetic_research_import_90days.csv'
import_df.to_csv(save_path, index=False)

file_size = os.path.getsize(save_path) / 1024

print("="*55)
print("  CSV File Saved")
print("="*55)
print(f"  File:  {save_path}")
print(f"  Rows:  {len(import_df):,}")
print(f"  Size:  {file_size:.1f} KB")
print()

# Verify by reading back
verify = pd.read_csv(save_path)
print(f"  Verification read: {len(verify):,} rows")

if len(verify) == len(import_df):
    print("  ✅ Row count matches")
else:
    print("  ❌ Row count mismatch — check file")

print()
print("  Control totals for reconciliation:")
print(f"  Total quantity: {import_df['QuantityKg'].sum():.2f} kg")
print(f"  Grades:         {import_df['TeaGrade'].nunique()}")
print(f"  Date range:     {import_df['OriginalTransactionDate'].min()}")
print(f"                  to {import_df['OriginalTransactionDate'].max()}")
print()
print("  Per grade totals:")
for g in sorted(import_df['TeaGrade'].unique()):
    gdf = import_df[import_df['TeaGrade'] == g]
    print(f"    {g:<6}: {gdf['QuantityKg'].sum():>10.2f} kg  "
          f"({len(gdf)} rows)")
print()
print("="*55)
print("  SAVE THESE CONTROL TOTALS")
print("  You will need them during import approval")
print("="*55)
print()
print("✅ CSV saved and verified!")
print()
print("NEXT STEP:")
print("  Go to your SmartTea admin system")
print("  Admin → Warehouse Operations")
print("         → Operational Data Imports")
print("  Upload this file:")
print(f"  {os.path.abspath(save_path)}")

  CSV File Saved
  File:  data/synthetic_research_import_90days.csv
  Rows:  455
  Size:  42.7 KB

  Verification read: 455 rows
  ✅ Row count matches

  Control totals for reconciliation:
  Total quantity: 106709.10 kg
  Grades:         5
  Date range:     2023-10-02
                  to 2023-12-31

  Per grade totals:
    BOP   :   31032.60 kg  (91 rows)
    BOPF  :   26466.70 kg  (91 rows)
    DUST  :   22543.10 kg  (91 rows)
    FNGS  :   16406.00 kg  (91 rows)
    OP    :   10260.70 kg  (91 rows)

  SAVE THESE CONTROL TOTALS
  You will need them during import approval

✅ CSV saved and verified!

NEXT STEP:
  Go to your SmartTea admin system
  Admin → Warehouse Operations
         → Operational Data Imports
  Upload this file:
  C:\Users\mrmra\Desktop\Smart tea\SmartTea_AI\data\synthetic_research_import_90days.csv


In [6]:
# ============================================================
# Cell 6 — Regenerate CSV with correct controlled headers
# Headers must exactly match the system template
# ============================================================

rows_fixed = []

for _, r in df_90.iterrows():
    if r['DemandKg'] > 0:
        rows_fixed.append({
            'SourceSystem':               'SYNTHETIC-RESEARCH',
            'SourceRecordId':             (
                f"SYN-{r['TeaGrade']}-"
                f"{r['Date'].strftime('%Y%m%d')}"),
            'OriginalTransactionDate':    r['Date'].strftime(
                                              '%Y-%m-%d'),
            'TeaGrade':                   r['TeaGrade'],
            'ItemCode':                   f"TEA-{r['TeaGrade']}",
            'Quantity':                   round(r['DemandKg'], 2),
            'Unit':                       'kg',
            'TransactionType':            'ProductionOutput',
            'SourceReferenceNumber':      (
                f"SYN-PROD-{r['TeaGrade']}-"
                f"{r['Date'].strftime('%Y%m%d')}"),
            'SupplierOrProductionReference': (
                'SmartTea-Synthetic-Research'),
            'WarehouseCode':              'WH-MAIN',
            'BinCode':                    'BIN-01',
            'UnitCost':                   0.00,
            'Reason':                     (
                'Synthetic research data — '
                'AI model demonstration'),
        })

fixed_df = pd.DataFrame(rows_fixed)

# Verify column order matches template exactly
expected_cols = [
    'SourceSystem',
    'SourceRecordId',
    'OriginalTransactionDate',
    'TeaGrade',
    'ItemCode',
    'Quantity',
    'Unit',
    'TransactionType',
    'SourceReferenceNumber',
    'SupplierOrProductionReference',
    'WarehouseCode',
    'BinCode',
    'UnitCost',
    'Reason'
]

fixed_df = fixed_df[expected_cols]

print("="*55)
print("  Fixed CSV — Column Check")
print("="*55)
print()
print("  Columns in correct order:")
for i, col in enumerate(fixed_df.columns):
    print(f"  {i+1:>2}. {col}")

print()
print(f"  Total rows: {len(fixed_df):,}")
print()
print("  Sample row 1:")
for col in fixed_df.columns:
    print(f"    {col:<35}: {fixed_df.iloc[0][col]}")
print()
print("✅ Column check complete!")

  Fixed CSV — Column Check

  Columns in correct order:
   1. SourceSystem
   2. SourceRecordId
   3. OriginalTransactionDate
   4. TeaGrade
   5. ItemCode
   6. Quantity
   7. Unit
   8. TransactionType
   9. SourceReferenceNumber
  10. SupplierOrProductionReference
  11. WarehouseCode
  12. BinCode
  13. UnitCost
  14. Reason

  Total rows: 455

  Sample row 1:
    SourceSystem                       : SYNTHETIC-RESEARCH
    SourceRecordId                     : SYN-BOP-20231002
    OriginalTransactionDate            : 2023-10-02
    TeaGrade                           : BOP
    ItemCode                           : TEA-BOP
    Quantity                           : 372.4
    Unit                               : kg
    TransactionType                    : ProductionOutput
    SourceReferenceNumber              : SYN-PROD-BOP-20231002
    SupplierOrProductionReference      : SmartTea-Synthetic-Research
    WarehouseCode                      : WH-MAIN
    BinCode                            :

In [7]:
# Save the fixed CSV
save_path = 'data/synthetic_research_import_FIXED.csv'
fixed_df.to_csv(save_path, index=False)

file_size = os.path.getsize(save_path) / 1024

# Verify by reading back
verify = pd.read_csv(save_path)

print("="*55)
print("  Fixed CSV Saved")
print("="*55)
print(f"  File:  {save_path}")
print(f"  Rows:  {len(verify):,}")
print(f"  Size:  {file_size:.1f} KB")
print()

# Control totals
total_qty = fixed_df['Quantity'].sum()
inbound   = fixed_df[
    fixed_df['TransactionType'] == 'ProductionOutput'
]['Quantity'].sum()
outbound  = fixed_df[
    fixed_df['TransactionType'].isin([
        'CustomerOrder', 'Damage', 'Transfer'
    ])
]['Quantity'].sum()

print("  Control totals:")
print(f"  Total rows:       {len(fixed_df):,}")
print(f"  Total quantity:   {total_qty:,.2f} kg")
print(f"  Inbound total:    {inbound:,.2f} kg")
print(f"  Outbound total:   {outbound:,.2f} kg")
print()
print("  Per grade:")
for g in sorted(fixed_df['TeaGrade'].unique()):
    gdf = fixed_df[fixed_df['TeaGrade'] == g]
    print(f"    {g:<6}: {gdf['Quantity'].sum():>10.2f} kg")
print()
print("="*55)
print("  WRITE THESE DOWN FOR IMPORT FORM:")
print("="*55)
print(f"  Expected rows:    455")
print(f"  Inbound kg:       {inbound:,.2f}")
print(f"  Outbound kg:      0.00")
print()
print("  Full file path:")
print(f"  {os.path.abspath(save_path)}")
print()
print("✅ Fixed CSV ready for upload!")

  Fixed CSV Saved
  File:  data/synthetic_research_import_FIXED.csv
  Rows:  455
  Size:  92.4 KB

  Control totals:
  Total rows:       455
  Total quantity:   106,709.10 kg
  Inbound total:    106,709.10 kg
  Outbound total:   0.00 kg

  Per grade:
    BOP   :   31032.60 kg
    BOPF  :   26466.70 kg
    DUST  :   22543.10 kg
    FNGS  :   16406.00 kg
    OP    :   10260.70 kg

  WRITE THESE DOWN FOR IMPORT FORM:
  Expected rows:    455
  Inbound kg:       106,709.10
  Outbound kg:      0.00

  Full file path:
  C:\Users\mrmra\Desktop\Smart tea\SmartTea_AI\data\synthetic_research_import_FIXED.csv

✅ Fixed CSV ready for upload!


In [8]:
# Cell 8 — Fix all 3 validation errors
# Fix 1: Date format → ISO-8601 with Z
# Fix 2: TransactionType → ProductionReceipt
# Fix 3: Warehouse code (update after checking system)

# !! IMPORTANT: Check your warehouse code first !!
# Go to Admin system and find warehouse code
# Common values: WH-MAIN, MAIN, WH01, DEFAULT
# Update this variable with your real warehouse code:

WAREHOUSE_CODE = 'WH-MAIN'   # update this if different

rows_v3 = []

for _, r in df_90.iterrows():
    if r['DemandKg'] > 0:
        rows_v3.append({
            'SourceSystem':
                'SYNTHETIC-RESEARCH',
            'SourceRecordId':
                f"SYN-{r['TeaGrade']}-"
                f"{r['Date'].strftime('%Y%m%d')}",
            'OriginalTransactionDate':
                r['Date'].strftime('%Y-%m-%dT00:00:00Z'),
            'TeaGrade':
                r['TeaGrade'],
            'ItemCode':
                f"TEA-{r['TeaGrade']}",
            'Quantity':
                round(r['DemandKg'], 2),
            'Unit':
                'kg',
            'TransactionType':
                'ProductionReceipt',
            'SourceReferenceNumber':
                f"SYN-PROD-{r['TeaGrade']}-"
                f"{r['Date'].strftime('%Y%m%d')}",
            'SupplierOrProductionReference':
                'SmartTea-Synthetic-Research',
            'WarehouseCode':
                WAREHOUSE_CODE,
            'BinCode':
                'BIN-01',
            'UnitCost':
                0.00,
            'Reason':
                'Synthetic research data — '
                'AI model demonstration',
        })

v3_df = pd.DataFrame(rows_v3)

expected_cols = [
    'SourceSystem',
    'SourceRecordId',
    'OriginalTransactionDate',
    'TeaGrade',
    'ItemCode',
    'Quantity',
    'Unit',
    'TransactionType',
    'SourceReferenceNumber',
    'SupplierOrProductionReference',
    'WarehouseCode',
    'BinCode',
    'UnitCost',
    'Reason'
]
v3_df = v3_df[expected_cols]

print("="*55)
print("  Version 3 — All Fixes Applied")
print("="*55)
print()
print("  Fix 1 — Date format:")
print(f"    {v3_df['OriginalTransactionDate'].iloc[0]}")
print()
print("  Fix 2 — Transaction type:")
print(f"    {v3_df['TransactionType'].iloc[0]}")
print()
print("  Fix 3 — Warehouse code:")
print(f"    {v3_df['WarehouseCode'].iloc[0]}")
print()
print("  Fix 4 — Item codes:")
for g in sorted(v3_df['TeaGrade'].unique()):
    code = v3_df[v3_df['TeaGrade']==g]['ItemCode'].iloc[0]
    print(f"    {g}: {code}")
print()
print(f"  Total rows: {len(v3_df):,}")
print()
print("✅ Version 3 ready!")
print()
print("  Waiting for warehouse code confirmation")
print("  before saving...")

  Version 3 — All Fixes Applied

  Fix 1 — Date format:
    2023-10-02T00:00:00Z

  Fix 2 — Transaction type:
    ProductionReceipt

  Fix 3 — Warehouse code:
    WH-MAIN

  Fix 4 — Item codes:
    BOP: TEA-BOP
    BOPF: TEA-BOPF
    DUST: TEA-DUST
    FNGS: TEA-FNGS
    OP: TEA-OP

  Total rows: 455

✅ Version 3 ready!

  Waiting for warehouse code confirmation
  before saving...


In [9]:
# Cell 9 — Save Version 3 CSV
save_path_v3 = 'data/synthetic_research_import_V3.csv'
v3_df.to_csv(save_path_v3, index=False)

file_size = os.path.getsize(save_path_v3) / 1024
verify    = pd.read_csv(save_path_v3)

print("="*55)
print("  Version 3 CSV Saved")
print("="*55)
print(f"  File:  {save_path_v3}")
print(f"  Rows:  {len(verify):,}")
print(f"  Size:  {file_size:.1f} KB")
print()

total_qty = v3_df['Quantity'].sum()
print("  Control totals for import form:")
print(f"  Expected rows:    {len(v3_df)}")
print(f"  Inbound kg:       {total_qty:,.2f}")
print(f"  Outbound kg:      0.00")
print()
print("  Per grade:")
for g in sorted(v3_df['TeaGrade'].unique()):
    gdf = v3_df[v3_df['TeaGrade'] == g]
    print(f"    {g:<6}: {gdf['Quantity'].sum():>10.2f} kg")
print()
print("  Full path:")
print(f"  {os.path.abspath(save_path_v3)}")
print()

# Verify first row looks correct
print("  First row check:")
row = v3_df.iloc[0]
print(f"    Date:  {row['OriginalTransactionDate']}")
print(f"    Grade: {row['TeaGrade']}")
print(f"    Item:  {row['ItemCode']}")
print(f"    Type:  {row['TransactionType']}")
print(f"    WH:    {row['WarehouseCode']}")
print(f"    Qty:   {row['Quantity']} kg")
print()
print("✅ V3 CSV saved and ready for upload!")

  Version 3 CSV Saved
  File:  data/synthetic_research_import_V3.csv
  Rows:  455
  Size:  97.3 KB

  Control totals for import form:
  Expected rows:    455
  Inbound kg:       106,709.10
  Outbound kg:      0.00

  Per grade:
    BOP   :   31032.60 kg
    BOPF  :   26466.70 kg
    DUST  :   22543.10 kg
    FNGS  :   16406.00 kg
    OP    :   10260.70 kg

  Full path:
  C:\Users\mrmra\Desktop\Smart tea\SmartTea_AI\data\synthetic_research_import_V3.csv

  First row check:
    Date:  2023-10-02T00:00:00Z
    Grade: BOP
    Item:  TEA-BOP
    Type:  ProductionReceipt
    WH:    WH-MAIN
    Qty:   372.4 kg

✅ V3 CSV saved and ready for upload!
